In [2]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from tqdm import tqdm
import time
import os
import math
from afinn import Afinn
from collections import defaultdict
import polars.selectors as cs
import re
import concurrent.futures
import polars as pl

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/javclamar/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
def cargar_nrc_dict(path="../data/lexicons/NRC-Emotion-Lexicon-Wordlevel-v0.92.txt"):
    """
    Carga el diccionario NRC en memoria.
    """
    nrc_map = defaultdict(set)
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) == 3 and int(parts[2]) == 1:
                    nrc_map[parts[0]].add(parts[1])
    except FileNotFoundError:
        print(f"No se encontró el archivo NRC en {path}. NRC devolverá 0.")
    except Exception as e:
        print(f"Error cargando NRC: {e}")
    return nrc_map

def score_to_stars(score):
    """
    Toma los valores en el rango [-1, 1] y los convierte al rango [1, 5] (esto se hace para ver como de correlacionados estan stars con los scores)
    """
    return (score + 1) / 2 * 4 + 1

def calcular_nrc_stars(nrc_positive_density, nrc_negative_density):
    """
    Convierte las densidades positiva y negativa del NRC a un compound score [-1, 1]
    y luego lo transforma a la escala de 1 a 5 estrellas. 
    Esto se ha hecho para inlcuir a NRC a la evaluación junto con VADER y AFINN

    Args:
        nrc_positive_density (float): Densidad positiva asignada por NRC a una reseña
        nrc_negative_density (float): Densidad negativa asignada por NRC a una reseña

    """
    epsilon = 1e-6 
    
    nrc_compound = (nrc_positive_density - nrc_negative_density) / (nrc_positive_density + nrc_negative_density + epsilon)
        
    return score_to_stars(nrc_compound)
    
def calcular_vader(texts):
    """
    Calcula los scores de VADER para un conjunto de textos y devuelve el compound.

    Args:
        texts (
    """
    sia = SentimentIntensityAnalyzer()
    scores = [sia.polarity_scores(str(t))['compound'] for t in texts]
    return [score_to_stars(s) for s in scores]
    
def calcular_afinn(texts):
    """
    Calcula score AFINN normalizado tal como se hace en VADER para un conjunto de textos.
    """
    afinn = Afinn()
    scores = []
    alpha = 15
    
    for t in texts:
        raw = afinn.score(str(t))
        
        if raw != 0:
            norm = raw / math.sqrt((raw**2) + alpha)
        else:
            norm = 0.0
        scores.append(norm)
    return [score_to_stars(s) for s in scores]


def calcular_nrc(texts):
    """
    Calcula densidad de emociones NRC para un conjunto de textos y su score en estrellas.
    """
    emotions = ['anger', 'anticipation', 'disgust', 'fear', 'joy', 
                'sadness', 'surprise', 'trust', 'positive', 'negative']
    
    results = {f'nrc_{e}': [] for e in emotions}
    results['nrc_compound'] = [] 
    
    tokenizer = re.compile(r'\b[a-z]+\b')

    for text in texts:
        text_str = str(text).lower()
        
        words = tokenizer.findall(text_str)
        total_words = len(words)
        
        counts = {e: 0.0 for e in emotions}
        
        if total_words > 0:
            for w in words:
                if w in NRC_DICT:
                    found_emotions = NRC_DICT[w]
                    for e in found_emotions:
                        counts[e] += 1
            
            for e in emotions:
                counts[e] = counts[e] / total_words
                
        counts['compound'] = calcular_nrc_stars(counts['positive'], counts['negative'])
        
        for e in emotions:
            results[f'nrc_{e}'].append(counts[e])
            
        results['nrc_compound'].append(counts['compound'])
            
    return results

LEXICONS_AVAILABLE = {
    'vader_score': calcular_vader,
    'afinn_score': calcular_afinn,
    'nrc_emotions': calcular_nrc
}

NRC_DICT = cargar_nrc_dict() 

def process_chunk(args):
    """
    Función que ejecutan los nucleos del procesador para paralelizar el cálculo de los sentiment scores.
    """
    texts, active_models = args
    results = {}
    
    for model_name in active_models:
        if model_name in LEXICONS_AVAILABLE:
            try:
                results[model_name] = LEXICONS_AVAILABLE[model_name](texts)
            except Exception as e:
                print(f"Error en {model_name}: {e}")
                results[model_name] = [0.0] * len(texts)
                
    return results

In [4]:
test_texts = [
    "This place is amazing awesome great incredible best ever",
    "Terrible horrible disgusting worst nightmare ever",
]

for text in test_texts:
    result = calcular_nrc([text])
    print(f"\nTexto: {text}")
    print(f"  nrc_positive : {result['nrc_positive'][0]:.6f}")
    print(f"  nrc_negative : {result['nrc_negative'][0]:.6f}")
    print(f"  nrc_compound : {result['nrc_compound'][0]:.4f}")
    
    # Ver qué palabras reconoce el NRC
    words = re.findall(r'\b[a-z]+\b', text.lower())
    for w in words:
        if w in NRC_DICT:
            print(f"'{w}' → {NRC_DICT[w]}")
        else:
            print(f"'{w}' → NO ENCONTRADA en NRC")


Texto: This place is amazing awesome great incredible best ever
  nrc_positive : 0.000000
  nrc_negative : 0.000000
  nrc_compound : 3.0000
    'this' → NO ENCONTRADA en NRC
    'place' → NO ENCONTRADA en NRC
    'is' → NO ENCONTRADA en NRC
    'amazing' → NO ENCONTRADA en NRC
    'awesome' → NO ENCONTRADA en NRC
    'great' → NO ENCONTRADA en NRC
    'incredible' → NO ENCONTRADA en NRC
    'best' → NO ENCONTRADA en NRC
    'ever' → NO ENCONTRADA en NRC

Texto: Terrible horrible disgusting worst nightmare ever
  nrc_positive : 0.000000
  nrc_negative : 0.666667
  nrc_compound : 1.0000
    'terrible' → {'sadness', 'fear', 'negative', 'disgust', 'anger'}
    'horrible' → {'anger', 'negative', 'disgust', 'fear'}
    'disgusting' → {'anger', 'negative', 'disgust', 'fear'}
    'worst' → NO ENCONTRADA en NRC
    'nightmare' → {'fear', 'negative'}
    'ever' → NO ENCONTRADA en NRC


In [10]:
csv_reviews = '../data/csv/yelp_academic_dataset_review.csv'
csv_reviews_output_scores = '../results/sentiment_analysis/yelp_academic_dataset_review_scored.csv'
batch_size = 200000
total_rows = 6_990_280

def analyze_sentiment(input_csv, output_csv, lexicons=['vader_score', 'afinn_score', 'nrc_emotions']):
    """
    Calcular los sentiment scores para un csv de reviews según los lexicons dados

    Args:
        input_csv (str): Ruta del csv con las reviews
        output_csv (str): Ruta del csv donde se van a escribir las reviews con sus sentiment scores
        lexicons (list<str>): Lexicons con los que se va a cacular los sentiment scores
    """
    
    if os.path.exists(output_csv):
        try:
            df_check = pl.read_csv(output_csv, n_rows=1, ignore_errors=True)
            if df_check.height == 0:
                source_csv = input_csv
                temp_output_csv = output_csv
            else:
                print(f'Leyendo datos existentes de: {output_csv}')
                source_csv = output_csv
                temp_output_csv = output_csv + '.tmp'
        except Exception:
            source_csv = input_csv
            temp_output_csv = output_csv
    else:
        source_csv = input_csv
        temp_output_csv = output_csv

    dummy_output = process_chunk((['test'], lexicons))
    
    new_lexicon_cols = []
    for lexicon in lexicons:
        res = dummy_output[lexicon]
        if isinstance(res, dict):
            new_lexicon_cols.extend(sorted(res.keys()))
        else:
            new_lexicon_cols.append(lexicon)
            
    try:
        df_schema = pl.read_csv(source_csv, n_rows=1, ignore_errors=True)
        existing_columns = df_schema.columns
        
        final_columns = list(dict.fromkeys(existing_columns + new_lexicon_cols))
        
        with open(temp_output_csv, 'w') as f:
            f.write(','.join([f'{c}' for c in final_columns]) + '\n')
            
    except Exception as e:
        print(f'Error gestionando headers: {e}')
        return

    num_cores = os.cpu_count()
    
    reader = pl.read_csv_batched(source_csv, batch_size=batch_size, ignore_errors=True)
    start_time = time.time()
    
    with concurrent.futures.ProcessPoolExecutor(max_workers=num_cores) as executor:
        with tqdm(total=total_rows, unit='reviews', desc='Procesando') as pbar:
            while True:
                batches = reader.next_batches(1)
                if not batches: break
                
                df_batch = batches[0]
                
                if 'text' not in df_batch.columns:
                    raise ValueError('El archivo fuente no tiene la columna "text" necesaria.')

                texts = df_batch['text'].to_list()
                
                chunk_size = math.ceil(len(texts) / num_cores)
                chunks = [texts[i:i + chunk_size] for i in range(0, len(texts), chunk_size)]
                worker_args = [(chunk, lexicons) for chunk in chunks]
                
                results_generator = executor.map(process_chunk, worker_args)
                
                batch_data_flat = {col: [] for col in new_lexicon_cols}
                
                for res_dict in results_generator:
                    for lexicon in lexicons:
                        output = res_dict[lexicon]
                        
                        if isinstance(output, dict):
                            for sub_col in output:
                                batch_data_flat[sub_col].extend(output[sub_col])
                        else:
                            batch_data_flat[lexicon].extend(output)
                
                df_scored = df_batch.with_columns(
                    [pl.Series(name=col, values=batch_data_flat[col], dtype=pl.Float64) for col in new_lexicon_cols]
                )
                
                df_scored.select(final_columns).write_csv(
                    file=open(temp_output_csv, 'a'),
                    include_header=False,
                    quote_style='always'
                )
                
                pbar.update(len(texts))

    print(f'Procesamiento finalizado en: {(time.time() - start_time) / 60:.2f} min')

    if os.path.exists(output_csv) and source_csv == output_csv:
        os.remove(output_csv)
        os.rename(temp_output_csv, output_csv)

analyze_sentiment(csv_reviews, csv_reviews_output_scores, lexicons=['vader_score', 'afinn_score'])

Leyendo datos existentes de: ../results/sentiment_analysis/yelp_academic_dataset_review_scored.csv


Procesando: 100%|█████████████████████████████████████████████████████████████████████████████| 6990280/6990280 [40:16<00:00, 2892.25reviews/s]


Procesamiento finalizado en: 40.29 min


In [11]:
csv_reviews_output_scores = '../results/sentiment_analysis/yelp_academic_dataset_review_scored.csv'

df_scored = pl.scan_csv(csv_reviews_output_scores, ignore_errors=True)

print(df_scored.columns)

print(df_scored.head(5).select(cs.numeric()).collect())

['review_id', 'user_id', 'business_id', 'stars', 'text', 'date', 'num_chars', 'vader_score', 'afinn_score', 'nrc_anger', 'nrc_anticipation', 'nrc_disgust', 'nrc_fear', 'nrc_joy', 'nrc_negative', 'nrc_positive', 'nrc_sadness', 'nrc_surprise', 'nrc_trust', 'nrc_compound']
shape: (5, 15)
┌───────┬───────────┬────────────┬────────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ stars ┆ num_chars ┆ vader_scor ┆ afinn_scor ┆ … ┆ nrc_sadnes ┆ nrc_surpr ┆ nrc_trust ┆ nrc_compo │
│ ---   ┆ ---       ┆ e          ┆ e          ┆   ┆ s          ┆ ise       ┆ ---       ┆ und       │
│ i64   ┆ i64       ┆ ---        ┆ ---        ┆   ┆ ---        ┆ ---       ┆ f64       ┆ ---       │
│       ┆           ┆ f64        ┆ f64        ┆   ┆ f64        ┆ f64       ┆           ┆ f64       │
╞═══════╪═══════════╪════════════╪════════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ 3     ┆ 513       ┆ 4.7194     ┆ 4.680336   ┆ … ┆ 0.009901   ┆ 0.019802  ┆ 0.029703  ┆ 3.666655  │
│ 5    

/tmp/ipykernel_26094/1362818348.py:5: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  print(df_scored.columns)
